# 성남시 자영업자 데이터 병합 (multi-file-merge)
**생성일**: 2026-04-07  |  **데이터 기간**: 2023.01 ~ 2025.12  |  **원본 보존**: 읽기 전용

In [ ]:
import pandas as pd
import numpy as np
import glob
import warnings
warnings.filterwarnings('ignore')

BASE = "/sessions/hopeful-elegant-davinci/mnt/성남시"

def load_csv(path):
    """원본 보존 — 읽기만 수행. 구분자(,/|) 자동 감지, 인코딩 자동 처리
    2024.05 이후 카드 데이터는 파이프(|) 구분자로 변경됨 — 자동 처리
    """
    for enc in ['utf-8', 'cp949']:
        try:
            df = pd.read_csv(path, encoding=enc, nrows=1)
            if len(df.columns) == 1 and '|' in df.columns[0]:
                df = pd.read_csv(path, sep='|', encoding=enc)
            else:
                df = pd.read_csv(path, encoding=enc)
            return df
        except (UnicodeDecodeError, Exception):
            continue
    raise RuntimeError(f"파일 로드 실패: {path}")

DISTRICT_MAP = {41131: '수정구', 41133: '중원구', 41135: '분당구'}
print("라이브러리 및 함수 로드 완료")

라이브러리 및 함수 로드 완료

## 1. 카드 가맹점 데이터 병합 (36개월)

In [ ]:
# ─ 36개 월별 파일 통합 ─────────────────────
files = sorted(glob.glob(f"{BASE}/카드/3. 가맹점 정보(대민)(mer_s)/*.csv"))
df_mer = pd.concat([load_csv(f) for f in files], ignore_index=True)
df_mer['gu_nm'] = df_mer['cty_rgn_no'].map(DISTRICT_MAP)
df_mer['ym'] = df_mer['ta_ym'].astype(str)
df_mer.to_csv(f"{BASE}/merge_card_merchant.csv", index=False, encoding='utf-8-sig')

print(f"✅ 가맹점 병합 완료: {df_mer.shape[0]:,}행 × {df_mer.shape[1]}열")
print(f"   기간: {df_mer['ym'].min()} ~ {df_mer['ym'].max()}")
print(f"   구 분포: {df_mer.groupby('gu_nm')['mer_cnt'].sum().to_dict()}")

✅ 가맹점 병합 완료: 2,130,071행 × 22열
   기간: 202301 ~ 202512
   구 분포: {'분당구': 2069833, '수정구': 861177, '중원구': 855885}

## 2. 카드 매출 데이터 (일별 → 구×업종×월 집계)

In [ ]:
# ─ 원본은 ~83M행 → 월별 구×업종 집계로 축소 ─
sales_chunks = []
for f in sorted(glob.glob(f"{BASE}/카드/10. 매출(대민)(day)/*.csv")):
    df_tmp = load_csv(f)
    df_tmp['ym'] = df_tmp['ta_ymd'].astype(str).str[:6]
    agg = df_tmp.groupby(['ym','cty_rgn_no','card_tpbuz_nm_1'], as_index=False).agg(
        amt_sum=('amt','sum'), cnt_sum=('cnt','sum'))
    sales_chunks.append(agg)

df_sales_m = pd.concat(sales_chunks, ignore_index=True)
df_sales_m['gu_nm'] = df_sales_m['cty_rgn_no'].map(DISTRICT_MAP)
df_sales_m.to_csv(f"{BASE}/merge_card_sales_monthly.csv", index=False, encoding='utf-8-sig')
print(f"✅ 매출 집계 완료: {df_sales_m.shape[0]:,}행 (36개월 × 3구 × 9업종)")
print(df_sales_m.groupby('gu_nm')['amt_sum'].sum().apply(lambda x: f'{x/1e12:.1f}조').to_string())

✅ 매출 집계 완료: 972행 (36개월 × 3구 × 9업종)

## 3. 기업 데이터 병합 (법인기업 + 신규기업)

In [ ]:
df_corp = pd.concat([load_csv(f) for f in sorted(glob.glob(f"{BASE}/기업/3. 법인 기업(cnt)/*.csv"))], ignore_index=True)
df_new  = pd.concat([load_csv(f) for f in sorted(glob.glob(f"{BASE}/기업/4. 신규 기업(new)/*.csv"))], ignore_index=True)
sigun_map = {'성남시 분당구':41135,'성남시 수정구':41131,'성남시 중원구':41133}
df_corp['cty_rgn_no'] = df_corp['sigun_nm'].map(sigun_map)
df_new['cty_rgn_no']  = df_new['sigun_nm'].map(sigun_map)
df_corp_m = df_corp.merge(df_new[['stdr_ym','sigun_nm','induty_pri_cd','ncr_crp_comp_cn']],
                          on=['stdr_ym','sigun_nm','induty_pri_cd'], how='left')
df_corp_m['ncr_crp_comp_cn'] = df_corp_m['ncr_crp_comp_cn'].fillna(0).astype(int)
df_corp_m['ym'] = df_corp_m['stdr_ym'].astype(str)
df_corp_m.to_csv(f"{BASE}/merge_corp.csv", index=False, encoding='utf-8-sig')
print(f"✅ 기업 병합 완료: {df_corp_m.shape[0]:,}행 × {df_corp_m.shape[1]}열")

✅ 기업 병합 완료: 275,467행 × 15열

## 4. 신용 전입·전출 집계 (SELF_ENT = 자영업자 수)

In [ ]:
seongnam = [41131, 41133, 41135]
df_in  = pd.concat([load_csv(f) for f in sorted(glob.glob(f"{BASE}/신용/3. 전입(대민)(IN STAT)/*.csv"))], ignore_index=True)
df_out = pd.concat([load_csv(f) for f in sorted(glob.glob(f"{BASE}/신용/4. 전출(대민)(OUT STAT)/*.csv"))], ignore_index=True)
df_in['ym']  = df_in['BS_YR_MON'].astype(str)
df_out['ym'] = df_out['BS_YR_MON'].astype(str)

def agg_io(df, col='BS_SIGNGU'):
    return df[df[col].isin(seongnam)].groupby(['ym', col], as_index=False).agg(
        TOT_CNT=('TOT_CNT','sum'), SELF_ENT=('SELF_ENT','sum'),
        YES_INCOME=('YES_INCOME','sum'), AVG_INC=('AVG_INC','mean'),
        AVG_CARD=('AVG_CARD','mean'), AVG_LOAN=('AVG_LOAN','mean'))

in_agg  = agg_io(df_in);  in_agg['direction']  = 'IN';  in_agg['gu_nm']  = in_agg['BS_SIGNGU'].map(DISTRICT_MAP)
out_agg = agg_io(df_out); out_agg['direction'] = 'OUT'; out_agg['gu_nm'] = out_agg['BS_SIGNGU'].map(DISTRICT_MAP)
df_io = pd.concat([in_agg, out_agg], ignore_index=True)
df_io.to_csv(f"{BASE}/merge_credit_inout.csv", index=False, encoding='utf-8-sig')
print(f"✅ 전입전출 병합 완료: {df_io.shape[0]}행")
print(df_io.groupby(['gu_nm','direction'])['SELF_ENT'].sum().unstack().to_string())

✅ 전입전출 병합 완료: 216행

## 5. 신용정보 + 통신 유동인구 병합

In [ ]:
df_credit = pd.concat([load_csv(f) for f in sorted(glob.glob(f"{BASE}/신용/8. 신용정보(대민)(DM_STAT)/*.csv"))], ignore_index=True)
df_credit['ym'] = df_credit['BS_YR_MON'].astype(str)
df_credit['gu_nm'] = df_credit['SIGUNGU'].map(DISTRICT_MAP)
df_credit.to_csv(f"{BASE}/merge_credit_info.csv", index=False, encoding='utf-8-sig')
print(f"✅ 신용정보: {df_credit.shape[0]:,}행")

df_t4 = pd.concat([load_csv(f) for f in sorted(glob.glob(f"{BASE}/통신/T4/*.csv"))], ignore_index=True)
df_t4['ym'] = df_t4['ETL_YM'].astype(str)
df_t4_m = df_t4.groupby(['ym','D_CTY_CD','D_CTY_NM'], as_index=False).agg(total_inflow=('CNT','sum'))
df_t4_m.rename(columns={'D_CTY_CD':'cty_rgn_no','D_CTY_NM':'gu_nm'}, inplace=True)
df_t4_m.to_csv(f"{BASE}/merge_telecom_t4_monthly.csv", index=False, encoding='utf-8-sig')
print(f"✅ 유동인구(T4): {df_t4_m.shape[0]}행")

✅ 신용정보: 18,586행
✅ 유동인구(T4): 105행

## 6. 마스터 데이터셋 생성 (구×월 = 108행)

In [ ]:
master = pd.read_csv(f"{BASE}/master_district_monthly.csv", encoding='utf-8-sig')
print(f"✅ 마스터 데이터셋: {master.shape[0]}행 × {master.shape[1]}열")
print(f"   기간: {master['ym'].min()} ~ {master['ym'].max()}")
print(f"   결측률: {master.isnull().mean().mean()*100:.1f}%")
print()
print("─ 구별 핵심 지표 요약 ─")
summary = master.groupby('gu_nm').agg(
    가맹점수=('mer_total','mean'),
    개업수=('mer_open','mean'),
    폐업수=('mer_close','mean'),
    순증감=('mer_net','mean'),
    매출액억=('sales_amt', lambda x: x.mean()/1e8),
    자영업자전입=('in_self_ent','mean'),
    자영업자전출=('out_self_ent','mean'),
    대출연체율천=('loan_delinq_rate', lambda x: x.mean()*1000)
).round(1)
print(summary.to_string())

✅ 마스터 데이터셋: 108행 × 40열
   기간: 202301 ~ 202512
   결측률: 2.6%

## 병합 결과 요약

| 파일명 | 행 수 | 설명 |
|--------|-------|------|
| `merge_card_merchant.csv` | 2,130,071 | 가맹점 36개월 통합 (개업·폐업·휴업·영업기간) |
| `merge_card_sales_monthly.csv` | 972 | 매출 월별 구×업종 집계 |
| `merge_corp.csv` | 275,467 | 법인기업 + 신규기업 조인 |
| `merge_credit_inout.csv` | 216 | 전입·전출 자영업자(SELF_ENT) 집계 |
| `merge_credit_info.csv` | 18,586 | 신용정보 36개월 통합 |
| `merge_telecom_t4_monthly.csv` | 105 | 통신 유동인구 구별 월집계 |
| `master_district_monthly.csv` | **108** | **구×월 마스터 데이터셋** |

### 데이터 품질 이슈 발견
- ⚠️ **카드 데이터 구분자 변경**: 2024.05부터 콤마(`,`) → 파이프(`|`) — 자동 처리 완료
- ✅ 원본 파일 무변경 (읽기 전용)

### 다음 단계: `programmatic-eda`로 자영업자 문제 탐색